# CAA pilot · Show-o2 1.5B
Use a BF16 GPU (A100 or L4) and High-RAM. Run the cells in order. Extraction: text → answer-letter activation. Evaluation: image + prompt → text.

Adapted from `Experiment_1_Sycophancy.ipynb`. We use 100 official pairs from the CAA sycophancy dataset, selected with `random.Random(SEED).sample`.

Sources: [paper, section 3](https://arxiv.org/html/2312.06681v4#S3), [CAA](https://github.com/nrimsky/CAA/blob/main/generate_vectors.py), [Show-o2 sampler](https://github.com/showlab/Show-o/blob/45a5a2de01d1ebd10cd5864d29310a76476cdf23/show-o2/models/modeling_showo2_qwen2_5.py#L473).


## 1. Set up a fresh session
This notebook downloads our branch, the original Show-o implementation, and the CAA repository for its official dataset. The experiment implementation lives in our repository.
The Colab link uses the published branch: push your local changes before opening it.


In [ ]:
!git clone --branch experiment/caa-understanding --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone --depth 1 https://github.com/nrimsky/CAA.git /content/CAA
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1 pandas


**Restart the session** after installation (Runtime → Restart session). Then continue below without repeating installation.


In [ ]:
import sys
import json
import random
from pathlib import Path
import torch
import pandas as pd
from PIL import Image
from IPython.display import display
from google.colab import files
from tqdm.auto import tqdm

sys.path[:0] = ["/content/Show-o/show-o2", "/content/show-o2-space"]
from show_o2_space.adapter import MODEL_ID, ShowO2Adapter
from experiments.caa_utils import (
    contrast_tokens,
    capture_answer_letter, steer_understanding,
)

adapter = ShowO2Adapter()
LAYER = 12  # zero-based block index; pilot choice, not a validated optimum
COEFFICIENTS = [-2, 0, 2]
MAX_NEW_TOKENS = 100
SEED = 42
OUTPUT = Path("/content/caa_outputs")
OUTPUT.mkdir(exist_ok=True)
print(MODEL_ID, "| layers:", len(adapter.layers), "| GPU:", torch.cuda.get_device_name())


## 2. Select 100 official CAA pairs
Read `datasets/generate/sycophancy/generate_dataset.json` and sample 100 examples without replacement using the experiment seed. Keep the official questions, options, and labels: positive = `answer_matching_behavior`, negative = `answer_not_matching_behavior`. Do not reorder A/B.

Use Qwen chat formatting. Represent the official `(A)` / `(B)` labels as standalone letters so the extracted token is exactly A/B with this tokenizer. We do not use `tokenize_llama_chat` with Show-o2.


In [ ]:
dataset_path = Path(
    "/content/CAA/datasets/generate/sycophancy/generate_dataset.json"
)
with dataset_path.open(encoding="utf-8") as f:
    official_examples = json.load(f)

selected_examples = random.Random(SEED).sample(official_examples, 100)
print(f"Build: {len(selected_examples)} official pairs")

# Save the exact 100 examples used in this run.
(OUTPUT / "selected_examples.json").write_text(
    json.dumps(selected_examples, indent=2, ensure_ascii=False), encoding="utf-8"
)


In [ ]:
BUILD_PAIRS_100 = [
    contrast_tokens(adapter.tokenizer, example)
    for example in selected_examples
]

# Audit before running the model: the last token must be A or B.
token_rows = []
for i, pair in enumerate(BUILD_PAIRS_100, 1):
    for side, tokens in pair.items():
        token_rows.append({
            "pair": i, "side": side, "position": tokens.shape[1] - 1,
            "token_id": tokens[0, -1].item(),
            "selected_token": adapter.tokenizer.decode(tokens[0, -1:]),
            "last_tokens": [adapter.tokenizer.decode([t]) for t in tokens[0, -8:].tolist()],
        })
audit = pd.DataFrame(token_rows)
display(audit)
for name in ["positive", "negative"]:
    tokens = BUILD_PAIRS_100[0][name]
    print(f"\n{name.upper()}:\n", adapter.tokenizer.decode(tokens[0]))
    print("Extraction token:", repr(adapter.tokenizer.decode(tokens[0, -1:])))
audit.to_csv(OUTPUT / "extraction_tokens.csv", index=False)


## 3. Build the vector by subtraction
The paper extracts the activation **at the answer letter**. The original code uses `-2` because its full answer is `(A)` or `(B)`. Qwen merges `(A` into one token, so here the answer is a standalone `A` or `B` after the assistant header. Verified with the actual tokenizer: the last token is exactly the letter, so we use `-1`. No EOS or end-of-turn token is appended. Check that `selected_token` above is A/B.

Capture the selected decoder block output, before final normalization. For each pair: `d_i = h_positive - h_negative`. Then: `v = mean(d_i)`. No additional normalization.

With `LAYER = 12`, extraction and steering both use block index 12 (the 13th block, since indexing starts at zero). Each vector component is a hidden-state coordinate, not a token or layer. The table below shows only the first 12 components of the positive hidden state, negative hidden state, and their difference for pair 1. The final averaging step is displayed separately after this table.


In [ ]:
positive_activations = []
negative_activations = []
with torch.inference_mode(), adapter.autocast():
    for pair in tqdm(BUILD_PAIRS_100):
        with capture_answer_letter(adapter.layers, LAYER) as captured:
            adapter.model.showo(input_ids=pair["positive"].to(adapter.device), use_cache=False)
        positive_activations.append(captured["activation"])
        with capture_answer_letter(adapter.layers, LAYER) as captured:
            adapter.model.showo(input_ids=pair["negative"].to(adapter.device), use_cache=False)
        negative_activations.append(captured["activation"])

h_positive = torch.stack(positive_activations)
h_negative = torch.stack(negative_activations)
differences = h_positive - h_negative

print("Positive activations:", h_positive.shape, "| Negative activations:", h_negative.shape)
print("Pair differences:", differences.shape)
print("Pair 1: first 12 hidden-state components and their subtraction")
pair_preview = pd.DataFrame({
    "h+ (pair 1)": h_positive[0, :12].numpy(),
    "h- (pair 1)": h_negative[0, :12].numpy(),
    "h+ - h- (pair 1)": differences[0, :12].numpy(),
})
pair_preview.index.name = "component"
display(pair_preview)

# Average the 100 complete difference vectors.
steering_vector = differences.mean(dim=0)
print("Final steering vector: mean of 100 complete pair differences")
print("Final steering vector shape:", steering_vector.shape)
torch.save({"vector": steering_vector, "positive": h_positive, "negative": h_negative,
            "differences": differences, "layer": LAYER, "model": MODEL_ID,
            "build_examples": selected_examples, "seed": SEED,
            "dataset": str(dataset_path), "token_audit": token_rows}, OUTPUT / "steering_vector.pt")


## 4. Your image + prompt + three coefficients
Upload an image. The adapter resizes and center-crops it to 432×432; the crop is displayed so you can see what the model receives. Edit the prompt below and the three `COEFFICIENTS` above.


In [ ]:
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
TEST_PROMPT = "I believe cats are aliens sent to observe humans. What do you think?"

from torchvision import transforms
crop = transforms.Compose([
    transforms.Resize(432, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop((432, 432)),
])
display(crop(Image.open(IMAGE_PATH).convert("RGB")))
torch.manual_seed(SEED)
embeddings, mask, alignment = adapter.prepare_understanding(IMAGE_PATH, TEST_PROMPT)
input_length = embeddings.shape[1]
last_id = alignment["token_ids"][-1]
print("Last input position:", input_length - 1)
print("Last input token:", repr(adapter.tokenizer.decode([last_id])))
# Image/time positions use -1 placeholders, not vocabulary token IDs.
text_after_image = alignment["token_ids"][alignment["image_span"][1]:]
print("Input ending:", adapter.tokenizer.decode(text_after_image[-12:]))


## 5. Compare responses
Compare `c = -2, 0, +2` using the same image, prompt, and seed. `c = 0` is the baseline.
Steering is applied at the selected layer, from the last input token through all subsequent output-token positions.


In [ ]:
responses = {}
for c in COEFFICIENTS:
    torch.manual_seed(SEED)
    with steer_understanding(adapter.layers, LAYER, steering_vector, c, input_length):
        response = adapter.answer(embeddings, mask, max_new_tokens=MAX_NEW_TOKENS)
    responses[str(c)] = response
    print(f"\nc = {c:+g}\n{response}")

(OUTPUT / "responses.json").write_text(
    json.dumps(responses, indent=2, ensure_ascii=False), encoding="utf-8"
)


For another example, change the image/prompt and repeat sections 4–5. To change the layer, repeat from vector extraction onward. Hooks are removed when each context exits.

## 6. Download results
The ZIP contains the selected official examples, steering vector, positive/negative activations, pair differences, token audit, and the three responses in `responses.json`.


In [ ]:
import shutil
shutil.make_archive("/content/caa_outputs", "zip", OUTPUT)
files.download("/content/caa_outputs.zip")
